In [ ]:
import joblib
import numpy as np
import tensorflow as tf
from tensorflow.keras import layers, models, optimizers
from tensorflow.keras.callbacks import EarlyStopping
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from tensorflow.keras.models import load_model

import matplotlib.pyplot as plt
import seaborn as sns

# 1. Load the Trained Model
print("Loading the transfer learning model...")
model = load_model("cross_section_transfer_model.keras")
print("Model loaded successfully.\n")

# 2. Load the Feature Scaler
print("Loading the feature scaler...")
scaler = joblib.load("feature_scaler.joblib")
print(f"Scaler loaded. (Mean: {scaler.mean_}, Scale/Std: {scaler.scale_})\n")

# 3. Load the Training History
print("Loading training history...")
history_data = np.load("training_history.npz")
history_fit_loss = history_data['history_fit_loss']
history_fit_val_loss = history_data['history_fit_val_loss']
history_raw_loss = history_data['history_raw_loss']
history_raw_val_loss = history_data['history_raw_val_loss']
print(f"History loaded. Phase 1 (Fit) ran for {len(history_fit_loss)} epochs.")
print(f"History loaded. Phase 2 (Raw) ran for {len(history_raw_loss)} epochs.\n")

# 4. Load the Train/Test Datasets
print("Loading train and test datasets...")
train_test_data = np.load("train_test_data.npz")

# Extract Fitted data splits
Xfit_train = train_test_data['Xfit_train']
yfit_train = train_test_data['yfit_train']
Xfit_test = train_test_data['Xfit_test']
yfit_test = train_test_data['yfit_test']

# Extract Raw data splits
Xraw_train = train_test_data['Xraw_train']
yraw_train = train_test_data['yraw_train']
Xraw_test = train_test_data['Xraw_test']
yraw_test = train_test_data['yraw_test']

print("Datasets loaded successfully!")
print(f"  - Fitted Training X shape: {Xfit_train.shape}")
print(f"  - Raw Testing X shape: {Xraw_test.shape}")

In [ ]:
# check Xraw correlation plot
sns.heatmap(np.corrcoef(Xraw_test.T), annot=True, cmap='coolwarm')
plt.title('Correlation matrix of Xraw_test')
plt.show()

sns.heatmap(np.corrcoef(Xraw_train.T), annot=True, cmap='coolwarm')
plt.title('Correlation matrix of Xraw_train')
plt.show()

In [ ]:
# check Xfit correlation plot
sns.heatmap(np.corrcoef(Xfit_test.T), annot=True, cmap='coolwarm')
plt.title('Correlation matrix of Xfit_test')
plt.show()

sns.heatmap(np.corrcoef(Xfit_train.T), annot=True, cmap='coolwarm')
plt.title('Correlation matrix of Xfit_train')
plt.show()

In [ ]:
# plot history_data
plt.figure(figsize=(12, 5))
plt.subplot(1, 2, 1)
plt.plot(history_fit_loss, label='Fit Train Loss')
plt.plot(history_fit_val_loss, label='Fit Val Loss')
plt.title('Phase 1: Fit Loss')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.legend()    

In [ ]:
Xfit_train_scaled = scaler.transform(Xfit_train)
Xfit_test_scaled = scaler.transform(Xfit_test)
Xraw_train_scaled = scaler.transform(Xraw_train)
Xraw_test_scaled = scaler.transform(Xraw_test)

y_pred_fit_test = model.predict(Xfit_test_scaled)
y_pred_fit_train = model.predict(Xfit_train_scaled)
y_pred_raw_test = model.predict(Xraw_test_scaled)
y_pred_raw_train = model.predict(Xraw_train_scaled)

In [ ]:

import numpy as np

def calculate_metrics_in_chunks(y_true, y_pred, chunk_size=500000):
    """
    Calculates MAPE and RMSE in chunks, ensuring shapes match to prevent 
    broadcasting memory explosions.
    """
    # FIX: Flatten both arrays to ensure they are strictly 1D (shape: (N,))
    y_true = np.asarray(y_true).flatten()
    y_pred = np.asarray(y_pred).flatten()
    
    n = len(y_true)
    sum_sq_error = 0.0
    sum_abs_perc_error = 0.0
    
    for i in range(0, n, chunk_size):
        y_true_chunk = y_true[i : i + chunk_size]
        y_pred_chunk = y_pred[i : i + chunk_size]
        
        # Calculate error for this chunk
        error = y_true_chunk - y_pred_chunk
        
        # Accumulate the sum of squared errors
        sum_sq_error += np.sum(error ** 2)
        
        # Accumulate the sum of absolute percentage errors
        sum_abs_perc_error += np.sum(np.abs(error / y_true_chunk))
        
    final_rmse = np.sqrt(sum_sq_error / n)
    final_mape = (sum_abs_perc_error / n) * 100
    
    return final_mape, final_rmse

print(f"yfit_test: min={yfit_test.min()}, max={yfit_test.max()}")
print(f"y_pred_fit_test: min={y_pred_fit_test.min()}, max={y_pred_fit_test.max()}")

# --- Execute for Test Data ---
fit_test_mape, fit_test_rmse = calculate_metrics_in_chunks(yfit_test, y_pred_fit_test)
# --- Execute for Train Data ---
fit_train_mape, fit_train_rmse = calculate_metrics_in_chunks(yfit_train, y_pred_fit_train)
print(f"Phase 1 (Fit)  Test MAPE: {fit_test_mape:.2f}%")
print(f"Phase 1 (Fit) Train MAPE: {fit_train_mape:.2f}%")
print(f"Phase 1 (Fit)  Test RMSE: {fit_test_rmse:.4f}")
print(f"Phase 1 (Fit) Train RMSE: {fit_train_rmse:.4f}")

raw_test_mape, raw_test_rmse = calculate_metrics_in_chunks(yraw_test, y_pred_raw_test)
raw_train_mape, raw_train_rmse = calculate_metrics_in_chunks(yraw_train, y_pred_raw_train)
print(f"Phase 2 (Raw)  Test MAPE: {raw_test_mape:.2f}%")
print(f"Phase 2 (Raw) Train MAPE: {raw_train_mape:.2f}%")
print(f"Phase 2 (Raw)  Test RMSE: {raw_test_rmse:.4f}")
print(f"Phase 2 (Raw) Train RMSE: {raw_train_rmse:.4f}")